# 데이터 로드 및 병합 파이프라인

GitHub에서 미리보기가 가능하도록 구성된 주피터 노트북 파일입니다.

In [1]:
import pandas as pd
import os

DATA_DIR = "data"

## 1. 데이터 로드

In [2]:
def load_data():
    files = {
        '기본정보': '기본정보.csv',
        '면적정보': '면적정보.csv',
        '시설정보': '시설정보.csv',
        '운영정보': '운영정보.csv',
        '위치정보': '위치정보.csv',
        '장기수선': '장기수선.csv',
        '관리비': '관리비.csv'
    }
    dfs = {}
    for name, filename in files.items():
        path = os.path.join(DATA_DIR, filename)
        try:
            dfs[name] = pd.read_csv(path, encoding='utf-8')
        except UnicodeDecodeError:
            dfs[name] = pd.read_csv(path, encoding='cp949')
        print(f"[{name}] 데이터 로드 완료 (원본): {dfs[name].shape}")
    return dfs

dfs = load_data()

[기본정보] 데이터 로드 완료 (원본): (2454, 19)
[면적정보] 데이터 로드 완료 (원본): (13852, 7)
[시설정보] 데이터 로드 완료 (원본): (2454, 12)
[운영정보] 데이터 로드 완료 (원본): (2454, 12)
[위치정보] 데이터 로드 완료 (원본): (2454, 3)
[장기수선] 데이터 로드 완료 (원본): (147240, 10)


[관리비] 데이터 로드 완료 (원본): (147240, 38)


## 2. 불필요한 칼럼 일괄 삭제

In [3]:
def drop_unnecessary_columns(dfs):
    cols_to_drop = {
        '관리비': [
            '차량유지비', '지능형네트워크유지비', '재해예방비', '가스사용료(공용)', '가스사용료(전용)', 
            '기타', '제세공과금', '교육훈련비', '시설유지비', '안전점검비', '위탁관리수수료', 
            '급탕비(공용)', '수도료(공용)', 'TV수신료', '정화조오물수수료', '선관위운영비'
        ],
        '기본정보': ['시공사', '주택관리업자'],
        '시설정보': [
            '건물구조', '전기-수전용량', '전기-세대전기계약방식', '승강기관리-관리방식', 
            'CCTV대수', '부대복리시설', '홈네트워크'
        ],
        '운영정보': [
            '경비관리-계약업체', '청소관리-계약업체', '음식물 처리방법', '소독관리-계약업체', 
            '일반관리-관리방식', '경비관리-관리방식', '청소관리-관리방식', '소독관리-관리방식'
        ],
        '장기수선': ['입주자기여수익', '공동기여수익']
    }
    print("\n[진행] 불필요한 칼럼 삭제 시작...")
    for table_name, columns in cols_to_drop.items():
        if table_name in dfs:
            dfs[table_name] = dfs[table_name].drop(columns=columns, errors='ignore')
            print(f"[{table_name}] 지정된 칼럼 삭제 완료 -> 남은 컬럼 수: {dfs[table_name].shape[1]}")
    return dfs

dfs = drop_unnecessary_columns(dfs)


[진행] 불필요한 칼럼 삭제 시작...
[관리비] 지정된 칼럼 삭제 완료 -> 남은 컬럼 수: 22
[기본정보] 지정된 칼럼 삭제 완료 -> 남은 컬럼 수: 17
[시설정보] 지정된 칼럼 삭제 완료 -> 남은 컬럼 수: 5
[운영정보] 지정된 칼럼 삭제 완료 -> 남은 컬럼 수: 4
[장기수선] 지정된 칼럼 삭제 완료 -> 남은 컬럼 수: 8


## 3. 데이터 병합 (Merge)

In [4]:
def merge_data(dfs):
    if '면적정보' in dfs:
        dfs['면적정보'] = dfs['면적정보'].drop_duplicates(subset=['단지코드']).drop(columns=['주거전용면적(세부)', '세대수'], errors='ignore')

    print("\n[진행] 정적 테이블(단지코드 기준) 병합 시작...")
    static_df = dfs['기본정보']
    static_tables = ['면적정보', '시설정보', '운영정보', '위치정보']
    for table_name in static_tables:
        df_to_merge = dfs[table_name]
        cols_to_use = [col for col in df_to_merge.columns if col not in static_df.columns or col == '단지코드']
        static_df = pd.merge(static_df, df_to_merge[cols_to_use], on='단지코드', how='left')
    
    print(f"정적 데이터 병합 완료: {static_df.shape}")

    print("\n[진행] 시계열 테이블(단지코드, 발생년월 기준) 병합 시작...")
    ts_df = dfs['관리비']
    repair_df = dfs['장기수선']
    join_keys = ['단지코드', '발생년월(YYYYMM)']
    cols_to_use = [col for col in repair_df.columns if col not in ts_df.columns or col in join_keys]
    ts_df = pd.merge(ts_df, repair_df[cols_to_use], on=join_keys, how='left')
    
    print(f"시계열 데이터 병합 완료: {ts_df.shape}")

    print("\n[진행] 최종 전체 병합 (시계열 데이터 + 정적 데이터) 시작...")
    final_df = pd.merge(ts_df, static_df, on='단지코드', how='left')
    print(f"최종 Master 데이터 병합 완료: {final_df.shape}")
    return final_df

final_df = merge_data(dfs)


[진행] 정적 테이블(단지코드 기준) 병합 시작...
정적 데이터 병합 완료: (2454, 28)

[진행] 시계열 테이블(단지코드, 발생년월 기준) 병합 시작...
시계열 데이터 병합 완료: (147240, 28)

[진행] 최종 전체 병합 (시계열 데이터 + 정적 데이터) 시작...
최종 Master 데이터 병합 완료: (147240, 55)


## 4. 최종 결과 확인

In [5]:
# 데이터프레임 미리보기
final_df.head()

,단지코드,발생년월(YYYYMM),공용관리비계,인건비,제사무비,피복비,그밖의부대비용,청소비,경비비,소독비,...,주거전용면적(단지합계),승강기대수,총주차대수,최고층수,지하층수,일반관리-인원,경비관리-인원,청소관리-인원,위도,경도
0,A11007001,202101,53343583.0,21552672,248500.0,0,341860,9046000,18157174,250000,...,22444.8,5,315,16,3,5,4.0,4,37.573567,126.972151
1,A11007001,202102,53189158.0,21552614,248100.0,0,326160,9046000,18157174,250000,...,22444.8,5,315,16,3,5,4.0,4,37.573567,126.972151
2,A11007001,202103,53228835.0,21552616,228500.0,0,711840,9046000,18157174,250000,...,22444.8,5,315,16,3,5,4.0,4,37.573567,126.972151
3,A11007001,202104,51995807.0,20485440,243900.0,0,315660,9046000,18157174,250000,...,22444.8,5,315,16,3,5,4.0,4,37.573567,126.972151
4,A11007001,202105,52384391.0,20494660,231100.0,0,398410,9046000,18157174,250000,...,22444.8,5,315,16,3,5,4.0,4,37.573567,126.972151


In [6]:
# 남은 최종 컬럼 리스트
print(list(final_df.columns))

['단지코드', '발생년월(YYYYMM)', '공용관리비계', '인건비', '제사무비', '피복비', '그밖의부대비용', '청소비', '경비비', '소독비', '승강기유지비', '수선비', '개별사용료계', '난방비(공용)', '난방비(전용)', '급탕비(전용)', '전기료(공용)', '전기료(전용)', '수도료(전용)', '생활폐기물수수료', '입대의운영비', '건물보험료', '시도_x', '장충금 월부과액', '장충금 월사용액', '장충금 총적립금액', '장충금 적립률', '잡수입 월수입금액', '시도_y', '시군구', '동리', '단지명', '단지분류', '법정동주소', '도로명주소', '분양형태', '사용승인일', '동수', '세대수', '분양세대수', '임대세대수', '관리방식', '난방방식', '복도유형', '관리비부과면적', '주거전용면적(단지합계)', '승강기대수', '총주차대수', '최고층수', '지하층수', '일반관리-인원', '경비관리-인원', '청소관리-인원', '위도', '경도']
